# Warehouse Robot Navigation with Tabular Reinforcement Learning
## Q-learning (off-policy) vs SARSA (on-policy), both implemented from scratch

**Case study — Reinforcement Learning**

**Research question.** *How do Q-learning and SARSA compare when a warehouse robot must learn the
shortest collision-free route to a delivery point?*

**Dataset.** `zoya77/robot-path-planning-navigation-dataset` (Kaggle) --- 7,980 warehouse
snapshots, each a 5x5 occupancy grid with a start cell and a goal cell.

---

### What this notebook learns from, and what it does not

This is the single most important thing to understand about applying reinforcement learning to a
Kaggle dataset, and it is stated here rather than buried.

Q-learning and SARSA **do not learn from rows of data**. They learn by acting and observing the
reward that follows. A CSV cannot answer the question *"what would have happened if the robot had
turned left instead?"* --- it only records what did happen. Something must be able to answer that
counterfactual, and that something is an environment.

| | What it is | Where it comes from |
|---|---|---|
| **Layouts** | Obstacle maps, start and goal cells | The Kaggle file, unmodified |
| **Heuristic baseline** | A move-toward-goal controller | Recovered from the file's `selected_action` |
| **Environment** | A `step()` function: movement, collisions, rewards, termination | Written in this notebook |
| **Reward constants** | Step cost, collision penalty, goal reward | **Declared modelling assumptions** |

So the warehouses are real and the robot faces genuine obstacle configurations. The cost structure
is an assumption, stated openly in Section 3 and varied in a sensitivity analysis in Section 8.
That assumption is not incidental: it is what creates the effect this study measures.

### What the data audit found

Section 1 audits every column. Briefly: the occupancy grids, start/goal cells and action encoding
are sound, but `reward`, `path_length_so_far` and `converged_flag` carry no usable signal, and the
`episode_id`/`time_step` columns do not describe coherent trajectories. Those columns are excluded,
with the evidence shown rather than asserted.

### The hypothesis

Q-learning and SARSA converge to the *same* policy on a benign grid. The difference appears only
when exploration can be costly. In a warehouse it is: the shortest route runs tight along the
racking, and an exploratory move in a narrow aisle causes a collision.

The classical prediction (Sutton and Barto, *Cliff Walking*) is therefore:

- **Q-learning**, learning the value of the *greedy* policy, converges to the optimal tight route
  and accumulates **more collisions while learning**.
- **SARSA**, learning the value of the policy it actually follows --- exploration included ---
  converges to a route with **clearance**, giving **better online performance** but a slightly
  longer final path.

This requires no injected randomness. Cliff Walking is deterministic; epsilon-greedy exploration
alone produces the effect.

### Methodology note

The predecessor study in this repository compared a BiLSTM against a Transformer and reported a
single seed per model as its principal limitation. Tabular RL is cheap enough to fix that properly:
every result here is **30 independent seeds per algorithm, paired by seed**, reported with
confidence intervals and a paired non-parametric significance test.

## 0. Configuration

`SMOKE_TEST = True` shrinks the notebook to a ~1 minute end-to-end run for checking it executes;
set it to `False` for the real experiment.

In [ ]:
SMOKE_TEST = False    # True -> 3 seeds / 300 episodes, for a fast end-to-end check

# ---- experiment ---------------------------------------------------------------
SEED          = 42
N_SEEDS       = 30         # independent runs per algorithm, paired by seed
N_EPISODES    = 3_000      # training episodes per run
TILE          = 4          # compose a TILE x TILE arrangement of real 5x5 patches
N_WAREHOUSES  = 12         # further composed floors for the generality check
MAX_STEPS     = 600        # episode cap

# ---- agent hyperparameters (identical for both algorithms) ---------------------
ALPHA         = 0.10       # learning rate
GAMMA         = 0.99       # discount factor
EPS_START     = 1.00
EPS_END       = 0.05
EPS_DECAY_FRAC= 0.60       # fraction of training over which epsilon decays

# ---- reward structure: THESE ARE MODELLING ASSUMPTIONS, not data ---------------
R_STEP        = -1.0       # every move costs time
R_BUMP        = -5.0       # nudging the racking: robot stops, no damage
R_INCIDENT    = -100.0     # entering the pedestrian walkway: safety incident
R_GOAL        = +100.0     # delivery completed
SLIP_PROB     = 0.0        # wheel slip; 0 = deterministic, as Cliff Walking is

if SMOKE_TEST:
    N_SEEDS, N_EPISODES, N_WAREHOUSES = 3, 300, 3

DATA_DIR = "data/warehouse"
ART_DIR  = "artifacts/rl"

ACTIONS = ["up", "down", "left", "right"]     # recovered from the data in Section 1
N_ACTIONS = len(ACTIONS)
MOVES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # (d_row, d_col)

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'} | "
      f"{N_SEEDS} seeds x {N_EPISODES} episodes per algorithm")

In [ ]:
import json, math, os, random, time, warnings
from collections import deque, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
Path(ART_DIR).mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)

set_seed()
print(f"numpy {np.__version__} | pandas {pd.__version__}")

In [ ]:
# ---- plotting style ------------------------------------------------------------
# Colour is assigned by algorithm and never reassigned; the same two slots are used
# in every figure so the reader learns the mapping once.
SERIES  = {"Q-learning": "#2a78d6", "SARSA": "#eb6834", "Heuristic": "#1baf7a",
           "Optimal (BFS)": "#6b7a8f"}
INK     = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#8a8880"}
GRID_C, SURFACE = "#e3e2de", "#fcfcfb"
BLUE_SEQ = LinearSegmentedColormap.from_list("blue_seq",
    ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID_C, "axes.linewidth": 0.8, "axes.labelcolor": INK["secondary"],
    "axes.titlecolor": INK["primary"], "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK["secondary"], "ytick.color": INK["secondary"],
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "grid.color": GRID_C, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK["secondary"],
    "lines.linewidth": 2.0, "lines.markersize": 7,
    "font.size": 10, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})

def recede(ax, axis="y"):
    ax.grid(axis=axis, zorder=0); ax.set_axisbelow(True); return ax

def save_fig(fig, name):
    p = Path(ART_DIR) / name
    fig.savefig(p); print(f"saved -> {p}")

## 1. Data acquisition and audit

The dataset is pulled through the Kaggle API so acquisition is reproducible. The newer `KGAT_`
token format is read from `~/.kaggle/access_token`.

The audit below is not a formality. Several columns in this file look meaningful and are not, and
using them would silently corrupt the study. Each exclusion is justified with evidence.

In [ ]:
def ensure_dataset(data_dir=DATA_DIR):
    csv = Path(data_dir) / "grid_robot_navigation_dataset.csv"
    if csv.exists():
        print(f"Found cached dataset at {csv}"); return csv
    token = Path.home() / ".kaggle" / "access_token"
    legacy = Path.home() / ".kaggle" / "kaggle.json"
    if not (token.exists() or legacy.exists() or os.environ.get("KAGGLE_API_TOKEN")):
        raise RuntimeError(
            "No Kaggle credentials found. Either:\n"
            "  mkdir -p ~/.kaggle && echo KGAT_xxx > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token\n"
            "  export KAGGLE_API_TOKEN=KGAT_xxx\n"
            "Get a token from https://www.kaggle.com/settings")
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    api.dataset_download_files("zoya77/robot-path-planning-navigation-dataset",
                               path=data_dir, unzip=True, quiet=True)
    print(f"Downloaded to {data_dir}/")
    return csv

CSV = ensure_dataset()
raw = pd.read_csv(CSV)
GRID_COLS = [f"obs_grid_flat_{i}" for i in range(25)]
print(f"\nRaw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

In [ ]:
# ---- Audit 1: are the 'episodes' real trajectories? ----------------------------
dup = int(raw.duplicated(["episode_id", "time_step"]).sum())
by_ep = raw.groupby("episode_id")
grid_varies  = not bool((by_ep[GRID_COLS].nunique().max(axis=1) == 1).all())
start_varies = not bool((by_ep[["start_x", "start_y"]].nunique().max(axis=1) == 1).all())

print("AUDIT 1 - do episode_id / time_step describe real trajectories?")
print(f"  rows duplicating the (episode_id, time_step) key : {dup:,} of {len(raw):,}")
print(f"  obstacle grid changes within a single episode    : {grid_varies}")
print(f"  start cell changes within a single episode       : {start_varies}")
print("  => VERDICT: these are independent snapshots, not trajectories.")
print("     episode_id and time_step are excluded.\n")

# ---- Audit 2: do reward / path_length carry task signal? -----------------------
def bfs_dist(grid, start, goal):
    # Exact shortest path on a unit-cost 4-connected grid; -1 if unreachable.
    if start == goal: return 0
    H, W = grid.shape
    q, seen = deque([(start, 0)]), {start}
    while q:
        (r, c), d = q.popleft()
        for dr, dc in MOVES.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and grid[nr, nc] == 0 and (nr, nc) not in seen:
                if (nr, nc) == goal: return d + 1
                seen.add((nr, nc)); q.append(((nr, nc), d + 1))
    return -1

grids5 = raw[GRID_COLS].to_numpy().reshape(-1, 5, 5)
starts = list(zip(raw.start_y, raw.start_x))
goals  = list(zip(raw.goal_y,  raw.goal_x))
true_d = np.array([bfs_dist(grids5[i], starts[i], goals[i]) for i in range(len(raw))])
ok = true_d >= 0

print("AUDIT 2 - do the supplied reward / path columns carry task signal?")
for col in ["reward", "path_length_so_far"]:
    r = np.corrcoef(raw[col].to_numpy()[ok], true_d[ok])[0, 1]
    print(f"  corr({col:<20}, true BFS distance) = {r:+.4f}")
print(f"  converged_flag unique values: {raw.converged_flag.unique().tolist()}")
print("  => VERDICT: no correlation with the actual task. All three excluded.")
print("     Ground truth comes from BFS, computed here and verified in Section 5.\n")

# ---- Audit 3: what the grids DO give us ----------------------------------------
on_obstacle = sum(grids5[i][starts[i]] or grids5[i][goals[i]] for i in range(len(raw)))
print("AUDIT 3 - the occupancy grids, start and goal cells")
print(f"  distinct 5x5 layouts            : {raw[GRID_COLS].drop_duplicates().shape[0]:,}")
print(f"  mean obstacles per 5x5 grid     : {grids5.sum(axis=(1,2)).mean():.2f} of 25")
print(f"  start or goal placed on a shelf : {on_obstacle}  (must be 0)")
print(f"  layouts with a reachable goal   : {ok.mean():.1%}")
print("  => VERDICT: sound. These are the usable content of the dataset.")

In [ ]:
# ---- Audit 4: recover the action encoding from the data ------------------------
# selected_action is an integer 0-3 with no documented meaning. If the logged controller
# moves toward the goal, the mean offset to the goal under each action reveals the encoding.
print("AUDIT 4 - recovering the action encoding")
print(f"  action counts: {np.bincount(raw.selected_action).tolist()}")
dy, dx = raw.delta_y.to_numpy(), raw.delta_x.to_numpy()
recovered = {}
for a in range(N_ACTIONS):
    m = raw.selected_action.to_numpy() == a
    mdy, mdx = dy[m].mean(), dx[m].mean()
    name = ("up" if mdy < -0.5 else "down" if mdy > 0.5 else
            "left" if mdx < -0.5 else "right" if mdx > 0.5 else "?")
    recovered[a] = name
    print(f"  action {a}: mean offset to goal (dy={mdy:+.2f}, dx={mdx:+.2f})  ->  {name}")

assert [recovered[a] for a in range(N_ACTIONS)] == ACTIONS, \
    f"recovered encoding {recovered} does not match assumed {ACTIONS}"
print(f"  => Encoding confirmed: {ACTIONS}")
print("     The logged controller moves toward the goal; it is used as a baseline in Section 7.")

# Long-term distance statistics, used to justify the composed warehouse in Section 2.
print(f"\n  BFS distance on the raw 5x5 layouts: median {np.median(true_d[ok]):.0f}, "
      f"90th pct {np.percentile(true_d[ok], 90):.0f}, max {true_d[ok].max()}")
print(f"  layouts needing 6+ steps: {(true_d[ok] >= 6).mean():.1%}")

## 2. Building the warehouse floor

### Why the raw 5x5 grids are not used directly

The audit found the supplied layouts are 5x5 with a median start-to-goal distance of 3 steps. That
is too small for the question being asked. The Q-learning/SARSA distinction is a *choice* between a
short risky route and a longer safe one, and at three steps there is no such choice: the shortest
path is the only path, both algorithms find it, and the comparison measures nothing.

This was confirmed empirically rather than assumed. On a floor built only from scattered racking,
both algorithms converge on the **identical** 39-step optimum. Scattered obstacles make the whole
floor uniformly risky, so there is no safe-route-versus-risky-route decision anywhere in the state
space.

### The floor, and what is assumed

Real warehouses are built from repeating rack modules crossed by main aisles, with pedestrian
walkways that automated robots must keep clear of. That structure is modelled directly:

| Element | Source |
|---|---|
| Racking across the floor | **From the data** --- 16 real 5x5 patches sampled from the 5,045 distinct layouts |
| Three cleared rows through the middle | **Assumed** --- a main cross-aisle, as every warehouse has |
| The pedestrian walkway | **Assumed** --- the hazard the robot must avoid |
| Start and goal at the aisle ends | **Assumed** --- charging station to delivery point |

The three cleared rows are, from the top:

1. **Walkway** --- people. Entering it is a safety incident: `-100`, and the robot is returned to
   its charging station. The episode continues.
2. **Main aisle** --- the direct route from start to goal, running immediately alongside the
   walkway. Fast, but one careless step upward causes an incident.
3. **Safe lane** --- a parallel route one row further from people. Slightly longer, much safer.

This is the structure of Sutton and Barto's *Cliff Walking*, expressed in warehouse terms, and it
is stated as a design choice rather than presented as something the dataset provided. The racking
that fills the rest of the floor is real, and it is what the robot must still navigate around.

In [ ]:
def compose_floor(patch_pool, tile=TILE, rng=None):
    # Tile real 5x5 obstacle patches into a (5*tile) x (5*tile) warehouse floor.
    rng = rng or np.random.default_rng(SEED)
    idx = rng.choice(len(patch_pool), size=tile * tile, replace=False)
    rows = [np.concatenate([patch_pool[idx[r * tile + c]] for c in range(tile)], axis=1)
            for r in range(tile)]
    return np.concatenate(rows, axis=0).astype(np.int8)


def build_warehouse(patch_pool, tile=TILE, rng=None):
    # Racking from the data; the cross-aisle and walkway are the declared assumptions.
    grid = compose_floor(patch_pool, tile, rng)
    H_, W_ = grid.shape
    rw = H_ // 2                                  # walkway row
    grid[rw, :] = 0                               # walkway   (people)
    grid[rw + 1, :] = 0                           # main aisle (the direct route)
    grid[rw + 2, :] = 0                           # safe lane  (the detour)
    walkway = np.zeros_like(grid, dtype=bool)
    walkway[rw, :] = True
    start, goal = (rw + 1, 0), (rw + 1, W_ - 1)
    return grid, walkway, start, goal


def bfs_field(grid, source, blocked=None):
    # Distance from `source` over passable cells; -1 where unreachable.
    H_, W_ = grid.shape
    block = (grid == 1) if blocked is None else ((grid == 1) | blocked)
    dist = -np.ones((H_, W_), dtype=int)
    if block[source]: return dist
    dist[source] = 0
    q = deque([source])
    while q:
        r, c = q.popleft()
        for dr, dc in MOVES.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < H_ and 0 <= nc < W_ and not block[nr, nc] and dist[nr, nc] < 0:
                dist[nr, nc] = dist[r, c] + 1
                q.append((nr, nc))
    return dist


patch_pool = np.unique(grids5.reshape(len(grids5), -1), axis=0).reshape(-1, 5, 5)
WAREHOUSE, WALKWAY, START, GOAL = build_warehouse(patch_pool, TILE, np.random.default_rng(SEED))
H, W = WAREHOUSE.shape
# The optimal route the robot is allowed to take: racking blocks, walkway is forbidden.
OPTIMAL_LEN = int(bfs_field(WAREHOUSE, START, blocked=WALKWAY)[GOAL])
WALK_ROW = int(np.argwhere(WALKWAY.any(axis=1))[0][0])

print(f"Floor            : {H} x {W} = {H*W} cells, racking from {TILE*TILE} real 5x5 patches")
print(f"Racking (blocked): {int(WAREHOUSE.sum())} cells ({WAREHOUSE.mean():.1%})")
print(f"Walkway row      : {WALK_ROW}   (hazard)")
print(f"Main aisle row   : {WALK_ROW+1}   Safe lane row: {WALK_ROW+2}")
print(f"Start -> Goal    : {START} -> {GOAL}")
print(f"Optimal route    : {OPTIMAL_LEN} steps along the main aisle  <-- BFS ground truth")

In [ ]:
def draw_floor(ax, grid, walkway, start, goal, paths=None, title=""):
    canvas = np.where(walkway, 2, grid)          # 0 floor, 1 racking, 2 walkway
    ax.imshow(canvas, cmap=ListedColormap(["#f3f5f8", "#3d5a80", "#f7c8b4"]),
              interpolation="nearest", vmin=0, vmax=2)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)
    ax.add_patch(plt.Circle((start[1], start[0]), 0.42, color="#1baf7a", zorder=6))
    ax.add_patch(plt.Circle((goal[1], goal[0]), 0.42, color="#eda100", zorder=6))
    if paths:
        for name, path in paths.items():
            if not path: continue
            ys, xs = zip(*path)
            ax.plot(xs, ys, color=SERIES[name], lw=2.8, alpha=0.95, zorder=5,
                    solid_capstyle="round", label=f"{name} — {len(path)-1} steps")
    ax.set_title(title, fontsize=11)
    return ax

LEGEND = [Patch(facecolor="#3d5a80", label="Racking (from data)"),
          Patch(facecolor="#f7c8b4", label="Pedestrian walkway"),
          Patch(facecolor="#1baf7a", label="Start"),
          Patch(facecolor="#eda100", label="Goal")]

fig, ax = plt.subplots(figsize=(5.8, 5.8))
draw_floor(ax, WAREHOUSE, WALKWAY, START, GOAL,
           title=f"Warehouse floor {H}x{W} — optimal route {OPTIMAL_LEN} steps")
ax.legend(handles=LEGEND, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2)
fig.tight_layout(); save_fig(fig, "fig1_warehouse.png"); plt.show()

## 3. The environment, from scratch

A plain class with `reset()` and `step()`. No `gymnasium`, no RL library.

**State** --- the robot's cell, `s = row * W + col`, giving `H x W = 400` states. Asserted
bijective in Section 5.

**Actions** --- the four moves recovered from the data in Section 1.

**Rewards --- the modelling assumptions.** The dataset contains no usable reward signal
(Section 1), so the cost structure is chosen and declared:

| Event | Reward | Effect | Justification |
|---|---|---|---|
| Ordinary step | `-1` | moves | time costs money |
| Nudging the racking | `-5` | stays put | a bump, not an incident |
| Entering the walkway | `-100` | **returned to the charging station** | a safety incident |
| Reaching the goal | `+100` | episode ends | delivery completed |

The walkway is what makes the problem interesting. Travelling the main aisle is the shortest route,
but every cell of it sits directly below the walkway, so a single exploratory step upward is a
`-100` incident. The safe lane one row further down is two steps longer per trip but has no such
exposure. **Which of those two routes an algorithm settles on is the entire experiment.**

In [ ]:
class WarehouseEnv:
    # Tabular grid-world MDP. Deterministic unless slip_prob > 0.
    def __init__(self, grid, walkway, start, goal, r_step=R_STEP, r_bump=R_BUMP,
                 r_incident=R_INCIDENT, r_goal=R_GOAL, slip_prob=SLIP_PROB,
                 max_steps=MAX_STEPS, rng=None):
        self.grid, self.walkway = grid, walkway
        self.start, self.goal = start, goal
        self.H, self.W = grid.shape
        self.n_states, self.n_actions = self.H * self.W, N_ACTIONS
        self.r_step, self.r_bump = r_step, r_bump
        self.r_incident, self.r_goal = r_incident, r_goal
        self.slip_prob, self.max_steps = slip_prob, max_steps
        self.rng = rng or np.random.default_rng(SEED)

    def encode(self, cell): return cell[0] * self.W + cell[1]
    def decode(self, s):    return (s // self.W, s % self.W)

    def reset(self):
        self.pos, self.t = self.start, 0
        return self.encode(self.pos)

    def step(self, action):
        self.t += 1
        if self.slip_prob > 0 and self.rng.random() < self.slip_prob:
            action = int(self.rng.integers(self.n_actions))      # wheel slip
        dr, dc = MOVES[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc
        info = {}

        if not (0 <= nr < self.H and 0 <= nc < self.W) or self.grid[nr, nc] == 1:
            reward, done = self.r_bump, False                    # nudge: no movement
            info["bump"] = True
        elif self.walkway[nr, nc]:
            self.pos = self.start                                # incident: back to charger
            reward, done = self.r_incident, False
            info["incident"] = True
        else:
            self.pos = (nr, nc)
            if self.pos == self.goal:
                reward, done, info["goal"] = self.r_goal, True, True
            else:
                reward, done = self.r_step, False
        if self.t >= self.max_steps:
            done, info["timeout"] = True, True
        return self.encode(self.pos), reward, done, info


env = WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL)
print(f"Environment ready: {env.n_states} states x {env.n_actions} actions "
      f"= {env.n_states * env.n_actions:,} Q-values, episode cap {env.max_steps} steps")

## 4. The two algorithms, from scratch

Both are temporal-difference control methods. Both keep a table `Q[s,a]`, both act
epsilon-greedily, and both apply

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\text{target} - Q(s,a)\big].$$

**They differ in one expression --- the target.**

| | Target | Character |
|---|---|---|
| **Q-learning** | $r + \gamma \max_{a'} Q(s',a')$ | *off-policy* --- values the **best** next action, whatever the robot actually does |
| **SARSA** | $r + \gamma\,Q(s',a')$ with $a' \sim \varepsilon\text{-greedy}(s')$ | *on-policy* --- values the action it **will actually take** |

That is the whole study. Q-learning learns the value of the greedy policy while behaving
exploratorily, so the cost of its own exploration never enters its estimates --- it is willing to
walk the aisle directly beneath the walkway, because a *greedy* robot would never step up into it.
SARSA learns the value of the policy it is actually running, so the `-100` risk of an exploratory
step is priced in, and the safe lane starts to look cheaper.

The shared machinery lives in `TDControlAgent`; the subclasses override only `_target`.

In [ ]:
class TDControlAgent:
    # Shared: Q-table, epsilon-greedy behaviour, epsilon schedule. Subclasses set the target.
    name = "TD"

    def __init__(self, n_states, n_actions, alpha=ALPHA, gamma=GAMMA,
                 eps_start=EPS_START, eps_end=EPS_END, seed=SEED):
        self.Q = np.zeros((n_states, n_actions), dtype=np.float64)
        self.n_actions = n_actions
        self.alpha, self.gamma = alpha, gamma
        self.eps_start, self.eps_end, self.eps = eps_start, eps_end, eps_start
        self.rng = np.random.default_rng(seed)

    def set_epsilon(self, frac_done, decay_frac=EPS_DECAY_FRAC):
        f = min(1.0, frac_done / decay_frac) if decay_frac > 0 else 1.0
        self.eps = self.eps_start + f * (self.eps_end - self.eps_start)

    def act(self, s):
        if self.rng.random() < self.eps:
            return int(self.rng.integers(self.n_actions))
        return self.greedy(s)

    def greedy(self, s):
        q = self.Q[s]
        return int(self.rng.choice(np.flatnonzero(q == q.max())))   # random tie-break

    def _target(self, r, s_next, a_next, done):
        raise NotImplementedError

    def update(self, s, a, r, s_next, a_next, done):
        self.Q[s, a] += self.alpha * (self._target(r, s_next, a_next, done) - self.Q[s, a])


class QLearningAgent(TDControlAgent):
    # Off-policy: bootstraps from the best available next action.
    name = "Q-learning"
    def _target(self, r, s_next, a_next, done):
        return r if done else r + self.gamma * self.Q[s_next].max()


class SarsaAgent(TDControlAgent):
    # On-policy: bootstraps from the action actually selected for the next step.
    name = "SARSA"
    def _target(self, r, s_next, a_next, done):
        return r if done else r + self.gamma * self.Q[s_next, a_next]


print("Q-learning target:  r + gamma * max_a' Q[s', a']")
print("SARSA      target:  r + gamma * Q[s', a']   with a' ~ epsilon-greedy(s')")

In [ ]:
def run_episode(env, agent, learn=True, greedy=False):
    # One episode. SARSA must choose a' BEFORE the update and then actually take it;
    # sampling a fresh action after updating is the classic SARSA implementation bug.
    s = env.reset()
    a = agent.greedy(s) if greedy else agent.act(s)
    total_r, steps, bumps, incidents, reached = 0.0, 0, 0, 0, False
    path = [env.decode(s)]
    while True:
        s_next, r, done, info = env.step(a)
        a_next = (agent.greedy(s_next) if greedy else agent.act(s_next)) if not done else 0
        if learn:
            agent.update(s, a, r, s_next, a_next, done)
        total_r += r; steps += 1
        bumps += int(info.get("bump", False))
        incidents += int(info.get("incident", False))
        reached = reached or bool(info.get("goal", False))
        path.append(env.decode(s_next))
        if done: break
        s, a = s_next, a_next
    return {"return": total_r, "steps": steps, "bumps": bumps, "incidents": incidents,
            "success": reached, "path": path}


def train(agent_cls, env, n_episodes=N_EPISODES, seed=SEED):
    agent = agent_cls(env.n_states, env.n_actions, seed=seed)
    env.rng = np.random.default_rng(seed + 10_000)
    keys = ("return", "steps", "bumps", "incidents", "success")
    hist = {k: np.zeros(n_episodes) for k in keys}
    for ep in range(n_episodes):
        agent.set_epsilon(ep / max(1, n_episodes - 1))
        out = run_episode(env, agent, learn=True)
        for k in keys: hist[k][ep] = out[k]
    return agent, hist


def evaluate_greedy(agent, env):
    # The learned policy with exploration switched off -- what you would deploy.
    return run_episode(env, agent, learn=False, greedy=True)

print("Training harness ready. Both algorithms run through this identical loop.")

## 5. Correctness tests

A reinforcement learning bug rarely announces itself. A mis-signed reward, an off-by-one in the
state encoding, or SARSA bootstrapping from a freshly sampled action rather than the one it will
actually take --- each still produces a learning curve that goes up and to the right. The agent
learns *something*, and the comparison between the two algorithms silently stops measuring what it
claims to.

These checks therefore run before any experiment, and assert on failure.

The strongest is check 7. With `epsilon = 0` the behaviour policy is greedy, so
$\max_{a'}Q(s',a')$ and $Q(s',a')$ are **the same quantity** and the two update rules are
mathematically identical. If the agents do not then produce bit-identical Q-tables, what separates
them is an implementation defect rather than the on-policy/off-policy distinction.

In [ ]:
checks = {}
e = WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL)

# ---- 1. state encoding is bijective -------------------------------------------
codes = {e.encode((r, c)) for r in range(e.H) for c in range(e.W)}
rt = all(e.decode(e.encode((r, c))) == (r, c) for r in range(e.H) for c in range(e.W))
checks["State encoding bijective"] = (len(codes) == e.H * e.W and rt,
                                      f"{len(codes)} codes for {e.H*e.W} cells, round-trip ok")

# ---- 2. nudging the racking: no movement, correct penalty, no termination ------
wall = np.zeros((3, 3), dtype=np.int8); wall[0, 1] = 1
no_hz = np.zeros((3, 3), dtype=bool)
e2 = WarehouseEnv(wall, no_hz, (1, 1), (2, 2)); e2.reset()
before = e2.pos
_, r, done, info = e2.step(0)                        # up into the racking
checks["Bump: robot does not move"] = (e2.pos == before, f"stayed at {e2.pos}")
checks["Bump: penalty, episode continues"] = (r == R_BUMP and not done and info.get("bump"),
                                              f"reward={r}, done={done}")

# ---- 3. walkway incident: penalty and return to the charging station ----------
hz = np.zeros((3, 3), dtype=bool); hz[0, 1] = True
e3 = WarehouseEnv(np.zeros((3, 3), np.int8), hz, (1, 1), (2, 2)); e3.reset()
_, r, done, info = e3.step(0)                        # up into the walkway
checks["Incident: returned to start"] = (e3.pos == e3.start, f"pos={e3.pos}, start={e3.start}")
checks["Incident: penalty, episode continues"] = (
    r == R_INCIDENT and not done and info.get("incident"), f"reward={r}, done={done}")

# ---- 4. goal terminates and pays out once -------------------------------------
e4 = WarehouseEnv(np.zeros((3, 3), np.int8), no_hz, (2, 1), (2, 2)); e4.reset()
_, r, done, info = e4.step(3)
checks["Goal: reward and termination"] = (r == R_GOAL and done and info.get("goal"),
                                          f"reward={r}, done={done}")

# ---- 5. BFS ground truth against a hand-checked grid --------------------------
hand = np.array([[0,0,1,0,0],[1,0,1,0,1],[0,0,0,0,0],[0,1,1,1,0],[0,0,0,0,0]], dtype=np.int8)
checks["BFS matches hand-checked grid"] = (int(bfs_field(hand, (0,0))[4,4]) == 8,
                                           f"BFS={int(bfs_field(hand,(0,0))[4,4])}, expected 8")
unreach = np.ones((3, 3), np.int8); unreach[0,0] = unreach[2,2] = 0
checks["BFS reports unreachable as -1"] = (int(bfs_field(unreach, (0,0))[2,2]) == -1, "")

# ---- 6. the walkway is excluded from the optimal route ------------------------
free_opt = int(bfs_field(WAREHOUSE, START)[GOAL])          # ignoring the walkway
checks["Optimal route avoids the walkway"] = (OPTIMAL_LEN >= free_opt,
    f"legal route {OPTIMAL_LEN} steps vs {free_opt} if the walkway were usable")

In [ ]:
# ---- 7. THE DECISIVE CHECK: at epsilon = 0 the two algorithms are identical ----
# Greedy behaviour makes max_a' Q[s',a'] and Q[s',a'] the same quantity, so any
# divergence here is a bug and not the on-policy/off-policy distinction.
g_s, w_s, st_s, go_s = build_warehouse(patch_pool, 2, np.random.default_rng(7))

def train_greedy_only(cls, seed=3):
    env_ = WarehouseEnv(g_s, w_s, st_s, go_s, max_steps=200)
    ag = cls(env_.n_states, env_.n_actions, eps_start=0.0, eps_end=0.0, seed=seed)
    env_.rng = np.random.default_rng(seed + 10_000)
    for ep in range(300):
        ag.set_epsilon(ep / 299)                     # stays 0.0 throughout
        run_episode(env_, ag, learn=True)
    return ag.Q

max_diff = float(np.abs(train_greedy_only(QLearningAgent) - train_greedy_only(SarsaAgent)).max())
checks["eps=0: Q-learning and SARSA identical"] = (max_diff == 0.0, f"max|Q_Q - Q_S| = {max_diff:.2e}")

# ---- 8. both recover the optimal policy on a tiny open grid -------------------
open3, no_hz3 = np.zeros((3, 3), np.int8), np.zeros((3, 3), bool)
opt3 = int(bfs_field(open3, (0, 0))[2, 2])
for cls in (QLearningAgent, SarsaAgent):
    ag, _ = train(cls, WarehouseEnv(open3, no_hz3, (0,0), (2,2), max_steps=100),
                  n_episodes=1500, seed=11)
    ev = evaluate_greedy(ag, WarehouseEnv(open3, no_hz3, (0,0), (2,2), max_steps=100))
    checks[f"{cls.name}: optimal on a 3x3 open grid"] = (
        ev["success"] and ev["steps"] == opt3, f"{ev['steps']} steps vs optimal {opt3}")

# ---- 9. reported return equals the rewards the environment actually emitted ---
env_a = WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL)
ag_a = QLearningAgent(env_a.n_states, env_a.n_actions, seed=5)
s = env_a.reset(); a = ag_a.act(s); manual = 0.0
while True:
    s2, r, done, _ = env_a.step(a); manual += r
    a2 = ag_a.act(s2) if not done else 0
    ag_a.update(s, a, r, s2, a2, done)
    if done: break
    s, a = s2, a2
env_b = WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL)
ag_b = QLearningAgent(env_b.n_states, env_b.n_actions, seed=5)
rep = run_episode(env_b, ag_b, learn=True)["return"]
checks["Return accounting matches the environment"] = (abs(manual - rep) < 1e-9,
                                                       f"manual={manual}, reported={rep}")

# ---- report --------------------------------------------------------------------
print(f"{'check':<46} {'result':<7} detail")
print("-" * 104)
failed = sum(not ok_ for ok_, _ in checks.values())
for name, (ok_, detail) in checks.items():
    print(f"{name:<46} {'PASS' if ok_ else 'FAIL':<7} {detail}")
print("-" * 104)
assert failed == 0, f"{failed} correctness check(s) failed - fix before running the experiment."
print(f"All {len(checks)} checks passed. Environment and both agents are correct.")

## 6. The multi-seed experiment

Every result below averages **30 independent runs per algorithm**, and the runs are **paired**:
seed *k* trains a Q-learning agent and a SARSA agent on the same floor, with the same episode
budget, the same hyperparameters and the same epsilon schedule. Only the target in the update rule
differs.

Pairing matters. It removes between-seed variance from the comparison and licenses a paired
significance test, which is far more sensitive than comparing two independent groups.

Two different questions are asked of every run, and the central claim of the on-policy/off-policy
distinction is that they have **different answers**:

- **Online performance** --- reward accumulated *while learning*, exploration switched on. What it
  costs to train the robot on a live warehouse floor.
- **Deployed quality** --- the greedy policy evaluated at `epsilon = 0` after training. What you
  would actually ship.

In [ ]:
def run_study(grid, walkway, start, goal, n_seeds=N_SEEDS, n_episodes=N_EPISODES, tag=""):
    results = {cls.name: {"hist": [], "eval": []} for cls in (QLearningAgent, SarsaAgent)}
    agents_s0, t0 = {}, time.time()
    for seed in range(n_seeds):
        for cls in (QLearningAgent, SarsaAgent):
            agent, hist = train(cls, WarehouseEnv(grid, walkway, start, goal),
                                n_episodes=n_episodes, seed=seed)
            ev = evaluate_greedy(agent, WarehouseEnv(grid, walkway, start, goal))
            results[cls.name]["hist"].append(hist)
            results[cls.name]["eval"].append(ev)
            if seed == 0: agents_s0[cls.name] = agent
        if (seed + 1) % max(1, n_seeds // 5) == 0:
            print(f"  {tag}seed {seed+1}/{n_seeds}  ({time.time()-t0:.0f}s)", flush=True)
    print(f"  {tag}done in {time.time()-t0:.0f}s")
    return results, agents_s0


print(f"Training {N_SEEDS} seeds x 2 algorithms x {N_EPISODES} episodes")
RESULTS, AGENTS = run_study(WAREHOUSE, WALKWAY, START, GOAL)

In [ ]:
def per_seed_table(results):
    rows = []
    for algo, d in results.items():
        for seed, (h, ev) in enumerate(zip(d["hist"], d["eval"])):
            tail = slice(int(0.8 * len(h["return"])), None)   # post-decay behaviour
            rows.append({
                "algorithm": algo, "seed": seed,
                # online: what training itself cost
                "online_return": h["return"].mean(),
                "online_return_late": h["return"][tail].mean(),
                "train_incidents": h["incidents"].sum(),
                "train_bumps": h["bumps"].sum(),
                "train_success_rate": h["success"].mean(),
                # deployed: what you would ship
                "greedy_steps": ev["steps"] if ev["success"] else np.nan,
                "greedy_success": float(ev["success"]),
                "greedy_return": ev["return"],
                "greedy_incidents": ev["incidents"],
            })
    return pd.DataFrame(rows)

per_seed = per_seed_table(RESULTS)
summary = per_seed.groupby("algorithm").agg(
    online_return=("online_return", "mean"),
    online_return_late=("online_return_late", "mean"),
    train_incidents=("train_incidents", "mean"),
    greedy_success=("greedy_success", "mean"),
    greedy_steps=("greedy_steps", "mean")).round(2)

print(f"Optimal route (BFS ground truth): {OPTIMAL_LEN} steps\n")
print("Mean over seeds\n"); print(summary.to_string())

## 7. Statistical comparison

Because both algorithms run on the same seeds, every metric yields **30 paired differences**. The
comparison uses the **Wilcoxon signed-rank test**, which assumes no particular distribution and
suits the small, skewed samples RL runs produce. A paired *t*-test is reported alongside as a
cross-check, and a bootstrap interval gives the **size** of the effect rather than only its
significance.

In [ ]:
def paired_test(df, metric, a="SARSA", b="Q-learning", n_boot=10_000, seed=SEED):
    x = df[df.algorithm == a].sort_values("seed")[metric].to_numpy(float)
    y = df[df.algorithm == b].sort_values("seed")[metric].to_numpy(float)
    keep = ~(np.isnan(x) | np.isnan(y)); x, y = x[keep], y[keep]
    d = x - y
    if len(d) == 0 or np.allclose(d, 0):
        return {"metric": metric, "n": int(len(d)), "median_diff": 0.0, "ci": [0.0, 0.0],
                "wilcoxon_p": 1.0, "ttest_p": 1.0, "significant": False}
    rng = np.random.default_rng(seed)
    boot = np.median(d[rng.integers(0, len(d), (n_boot, len(d)))], axis=1)
    return {"metric": metric, "n": int(len(d)), "median_diff": float(np.median(d)),
            "ci": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
            "wilcoxon_p": float(stats.wilcoxon(x, y).pvalue),
            "ttest_p": float(stats.ttest_rel(x, y).pvalue),
            "significant": bool(stats.wilcoxon(x, y).pvalue < 0.05)}

METRICS = ["online_return", "online_return_late", "train_incidents",
           "greedy_steps", "greedy_success"]
tests = [paired_test(per_seed, m) for m in METRICS]

print(f"Paired comparison, SARSA minus Q-learning  (n = {tests[0]['n']} seeds)\n")
print(f"{'metric':<22}{'median diff':>13}{'95% CI':>24}{'Wilcoxon p':>13}{'t-test p':>11}  verdict")
print("-" * 105)
for t in tests:
    ci = f"[{t['ci'][0]:.2f}, {t['ci'][1]:.2f}]"
    print(f"{t['metric']:<22}{t['median_diff']:>13.2f}{ci:>24}"
          f"{t['wilcoxon_p']:>13.2e}{t['ttest_p']:>11.2e}  "
          f"{'significant' if t['significant'] else 'not significant'}")
print("-" * 105)
print("A positive difference means SARSA scores HIGHER on that metric. For returns and success")
print("that is better; for incidents and step counts it is worse. Read each in context.")

## 8. What the two algorithms actually learned

The statistics say *whether* they differ. These figures say *how*.

In [ ]:
ep = np.arange(1, N_EPISODES + 1)
def band(ax, algo, key, smooth=51):
    arr = np.vstack([h[key] for h in RESULTS[algo]["hist"]])
    if smooth > 1 and arr.shape[1] > smooth:
        k = np.ones(smooth) / smooth
        arr = np.vstack([np.convolve(row, k, mode="valid") for row in arr])
    x = ep[:arr.shape[1]]
    ax.fill_between(x, np.percentile(arr, 2.5, axis=0), np.percentile(arr, 97.5, axis=0),
                    color=SERIES[algo], alpha=0.18, lw=0, zorder=2)
    ax.plot(x, arr.mean(0), color=SERIES[algo], label=algo, zorder=3)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 3.9))
for algo in ("Q-learning", "SARSA"):
    band(axes[0], algo, "return"); band(axes[1], algo, "incidents")
axes[0].set(title="Episode return while learning", xlabel="episode", ylabel="return")
axes[1].set(title="Walkway incidents per episode", xlabel="episode", ylabel="incidents")
for ax in axes: ax.legend(loc="best"); recede(ax)
fig.suptitle(f"Mean of {N_SEEDS} seeds; shaded band = 95% range across seeds",
             x=0.005, ha="left", fontsize=9.5, color=INK["muted"], y=1.05)
fig.tight_layout(); save_fig(fig, "fig2_learning_curves.png"); plt.show()

In [ ]:
# Distance of the learned route from the walkway: the quantitative form of "clearance".
def walkway_clearance(path):
    return float(np.mean([abs(r - WALK_ROW) for r, _ in path])) if path else np.nan

paths, clearance, deployed = {}, {}, {}
for algo, agent in AGENTS.items():
    ev = evaluate_greedy(agent, WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL))
    paths[algo] = ev["path"] if ev["success"] else []
    clearance[algo] = walkway_clearance(paths[algo])
    deployed[algo] = ev

fig, axes = plt.subplots(1, 2, figsize=(11.4, 5.6))
for ax, algo in zip(axes, ("Q-learning", "SARSA")):
    ev = deployed[algo]
    draw_floor(ax, WAREHOUSE, WALKWAY, START, GOAL, {algo: paths[algo]},
               title=f"{algo} — {ev['steps'] if ev['success'] else 'did not reach goal'} steps, "
                     f"mean distance from walkway {clearance[algo]:.2f}")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f"Deployed greedy routes, seed 0   ·   optimal is {OPTIMAL_LEN} steps along the aisle",
             x=0.005, ha="left", fontsize=10, color=INK["secondary"], y=1.01)
fig.tight_layout(); save_fig(fig, "fig3_learned_routes.png"); plt.show()

for a in ("Q-learning", "SARSA"):
    print(f"  {a:<12} {deployed[a]['steps']:>4} steps   mean distance from walkway "
          f"{clearance[a]:.3f}   success={deployed[a]['success']}")

In [ ]:
# State values V(s) = max_a Q(s,a), masked where racking blocks the floor.
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.8))
Vs = {a: np.ma.masked_array(ag.Q.max(1).reshape(H, W), WAREHOUSE == 1) for a, ag in AGENTS.items()}
vmin = min(v.min() for v in Vs.values()); vmax = max(v.max() for v in Vs.values())
for ax, algo in zip(axes, ("Q-learning", "SARSA")):
    im = ax.imshow(Vs[algo], cmap=BLUE_SEQ, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.axhline(WALK_ROW, color="#eb6834", lw=2.0, zorder=4)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)
    ax.add_patch(plt.Circle((START[1], START[0]), 0.42, color="#1baf7a", zorder=6))
    ax.add_patch(plt.Circle((GOAL[1], GOAL[0]), 0.42, color="#eda100", zorder=6))
    ax.set_title(algo, fontsize=11)
cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
cb.set_label("V(s) = max_a Q(s,a)", color=INK["secondary"], fontsize=9); cb.outline.set_visible(False)
fig.suptitle("Learned state values — orange line marks the pedestrian walkway",
             x=0.005, ha="left", fontsize=10, color=INK["secondary"], y=1.0)
save_fig(fig, "fig4_value_heatmaps.png"); plt.show()

In [ ]:
# Paired per-seed differences for the headline metrics.
fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.6))
for ax, metric, nicer in zip(axes,
        ["online_return", "train_incidents", "greedy_steps"],
        ["Online return while learning", "Walkway incidents in training", "Deployed route length"]):
    x = per_seed[per_seed.algorithm == "SARSA"].sort_values("seed")[metric].to_numpy(float)
    y = per_seed[per_seed.algorithm == "Q-learning"].sort_values("seed")[metric].to_numpy(float)
    d = x - y
    ax.axhline(0, color=INK["muted"], lw=1.2, zorder=1)
    ax.scatter(np.arange(len(d)), d, s=34, zorder=3,
               color=[SERIES["SARSA"] if v > 0 else SERIES["Q-learning"] for v in d])
    med = np.nanmedian(d)
    ax.axhline(med, color=INK["primary"], ls="--", lw=1.4, zorder=2)
    ax.annotate(f"median {med:+.1f}", xy=(0, med), xytext=(3, 6), textcoords="offset points",
                fontsize=9, color=INK["primary"])
    ax.set(title=nicer, xlabel="seed", ylabel="SARSA − Q-learning"); recede(ax)
fig.suptitle("Paired per-seed differences — one point per seed, both algorithms on identical runs",
             x=0.005, ha="left", fontsize=9.5, color=INK["muted"], y=1.05)
fig.tight_layout(); save_fig(fig, "fig5_paired_differences.png"); plt.show()

## 9. Three checks on whether the result is real

The headline comparison is one floor at one hazard penalty. Three things could make it an
artefact, and each is tested directly.

1. **Is either algorithm any good at all?** Compared against the heuristic controller recovered
   from the dataset and against the BFS optimum.
2. **Is it specific to this one floor?** Repeated on independently composed warehouses.
3. **Is it an artefact of the hazard penalty I chose?** Swept across a range of penalties,
   including zero.

In [ ]:
# --- 9.1 baselines: the dataset's own controller, and the BFS optimum ----------
def heuristic_policy(env, seed=SEED):
    # The controller recovered from `selected_action` in Section 1: move toward the goal.
    rng = np.random.default_rng(seed)
    env.reset(); total, steps, bumps, inc = 0.0, 0, 0, 0
    path = [env.pos]
    while steps < env.max_steps:
        r, c = env.pos; gr, gc = env.goal
        cand = ([0] if gr < r else []) + ([1] if gr > r else []) + \
               ([2] if gc < c else []) + ([3] if gc > c else [])
        a = int(rng.choice(cand)) if cand else int(rng.integers(4))
        _, rew, done, info = env.step(a)
        total += rew; steps += 1
        bumps += int(info.get("bump", False)); inc += int(info.get("incident", False))
        path.append(env.pos)
        if done:
            return {"return": total, "steps": steps, "bumps": bumps, "incidents": inc,
                    "success": bool(info.get("goal", False)), "path": path}
    return {"return": total, "steps": steps, "bumps": bumps, "incidents": inc,
            "success": False, "path": path}

heur = heuristic_policy(WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL))
rows = [{"policy": "Optimal (BFS)", "steps": OPTIMAL_LEN, "success": 1.0,
         "incidents": 0, "return": R_GOAL + R_STEP * (OPTIMAL_LEN - 1)},
        {"policy": "Heuristic (from data)", "steps": heur["steps"] if heur["success"] else np.nan,
         "success": float(heur["success"]), "incidents": heur["incidents"], "return": heur["return"]}]
for algo in ("Q-learning", "SARSA"):
    sub = per_seed[per_seed.algorithm == algo]
    rows.append({"policy": algo, "steps": sub.greedy_steps.mean(),
                 "success": sub.greedy_success.mean(),
                 "incidents": sub.greedy_incidents.mean(), "return": sub.greedy_return.mean()})
baseline_tbl = pd.DataFrame(rows).set_index("policy").round(2)
print("Deployed-policy comparison (learned rows are means over seeds)\n")
print(baseline_tbl.to_string())

In [ ]:
# --- 9.2 does the result survive on independently composed floors? ------------
gen_rows, t0 = [], time.time()
for w in range(N_WAREHOUSES):
    g_, wk_, st_, go_ = build_warehouse(patch_pool, TILE, np.random.default_rng(1000 + w))
    opt_ = int(bfs_field(g_, st_, blocked=wk_)[go_])
    for cls in (QLearningAgent, SarsaAgent):
        ag, hist = train(cls, WarehouseEnv(g_, wk_, st_, go_),
                         n_episodes=max(600, N_EPISODES // 3), seed=w)
        ev = evaluate_greedy(ag, WarehouseEnv(g_, wk_, st_, go_))
        gen_rows.append({"floor": w, "algorithm": cls.name, "optimal": opt_,
                         "greedy_steps": ev["steps"] if ev["success"] else np.nan,
                         "greedy_success": float(ev["success"]),
                         "train_incidents": hist["incidents"].sum(),
                         "online_return": hist["return"].mean()})
    print(f"  floor {w+1}/{N_WAREHOUSES} ({time.time()-t0:.0f}s)", flush=True)

gen = pd.DataFrame(gen_rows)
gen["excess_steps"] = gen.greedy_steps - gen.optimal
gen_summary = gen.groupby("algorithm").agg(
    success=("greedy_success", "mean"), excess_steps=("excess_steps", "mean"),
    train_incidents=("train_incidents", "mean"), online_return=("online_return", "mean")).round(2)
print(f"\nGenerality check over {N_WAREHOUSES} independently composed floors\n")
print(gen_summary.to_string())
gs = gen[gen.algorithm == "SARSA"].sort_values("floor").train_incidents.to_numpy()
gq = gen[gen.algorithm == "Q-learning"].sort_values("floor").train_incidents.to_numpy()
gen_p = float(stats.wilcoxon(gs, gq).pvalue) if not np.allclose(gs, gq) else 1.0
print(f"\nPaired Wilcoxon on training incidents across floors: p = {gen_p:.3e}")

In [ ]:
# --- 9.3 sensitivity to the hazard penalty (a declared assumption) ------------
PENALTIES = [0.0, -10.0, -25.0, -50.0, -100.0, -200.0]
sens_rows, t0 = [], time.time()
n_sens_seeds = min(8, N_SEEDS)
for pen in PENALTIES:
    for seed in range(n_sens_seeds):
        for cls in (QLearningAgent, SarsaAgent):
            ag, hist = train(cls, WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL, r_incident=pen),
                             n_episodes=max(600, N_EPISODES // 3), seed=seed)
            ev = evaluate_greedy(ag, WarehouseEnv(WAREHOUSE, WALKWAY, START, GOAL, r_incident=pen))
            sens_rows.append({"penalty": pen, "seed": seed, "algorithm": cls.name,
                              "train_incidents": hist["incidents"].sum(),
                              "greedy_steps": ev["steps"] if ev["success"] else np.nan,
                              "greedy_success": float(ev["success"]),
                              "clearance": walkway_clearance(ev["path"]) if ev["success"] else np.nan})
    print(f"  penalty {pen:>7} done ({time.time()-t0:.0f}s)", flush=True)
sens = pd.DataFrame(sens_rows)

fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.8))
for algo in ("Q-learning", "SARSA"):
    s_ = sens[sens.algorithm == algo].groupby("penalty")
    for ax, col in zip(axes, ["train_incidents", "clearance"]):
        m, e = s_[col].mean(), s_[col].sem()
        ax.errorbar(m.index, m.values, yerr=e.values, marker="o", capsize=3,
                    color=SERIES[algo], label=algo)
axes[0].set(title="Walkway incidents during training", xlabel="hazard penalty", ylabel="incidents")
axes[1].set(title="Distance of the learned route from the walkway",
            xlabel="hazard penalty", ylabel="mean distance")
for ax in axes: ax.legend(loc="best"); recede(ax)
fig.tight_layout(); save_fig(fig, "fig6_penalty_sensitivity.png"); plt.show()
print(sens.groupby(["penalty", "algorithm"])[["train_incidents", "clearance", "greedy_steps"]]
      .mean().round(2).to_string())

In [ ]:
# --- persist every reported number ----------------------------------------------
metrics = {
    "config": {k: globals()[k] for k in
               ["SMOKE_TEST", "SEED", "N_SEEDS", "N_EPISODES", "TILE", "MAX_STEPS", "ALPHA",
                "GAMMA", "EPS_START", "EPS_END", "EPS_DECAY_FRAC", "R_STEP", "R_BUMP",
                "R_INCIDENT", "R_GOAL", "SLIP_PROB"]},
    "warehouse": {"shape": [int(H), int(W)], "racking_cells": int(WAREHOUSE.sum()),
                  "walkway_row": WALK_ROW, "start": list(map(int, START)),
                  "goal": list(map(int, GOAL)), "optimal_steps": OPTIMAL_LEN},
    "correctness_checks": {k: bool(v[0]) for k, v in checks.items()},
    "summary": summary.to_dict(),
    "paired_tests": tests,
    "clearance": clearance,
    "deployed": {a: {k: (float(v) if isinstance(v, (int, float, np.floating)) else bool(v))
                     for k, v in d.items() if k != "path"} for a, d in deployed.items()},
    "baselines": baseline_tbl.to_dict(),
    "generality": gen_summary.to_dict(), "generality_wilcoxon_p": gen_p,
    "sensitivity": sens.groupby(["penalty", "algorithm"])[
        ["train_incidents", "clearance", "greedy_steps", "greedy_success"]]
        .mean().round(4).reset_index().to_dict(orient="records"),
}
with open(Path(ART_DIR) / "metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2, default=float)
per_seed.to_csv(Path(ART_DIR) / "per_seed_results.csv", index=False)
print(f"Wrote {Path(ART_DIR)/'metrics.json'} and per_seed_results.csv")
print("Artifacts:", sorted(p.name for p in Path(ART_DIR).iterdir()))